# RAG over QASPER and PubMedQA — Group Code Report

**Project:** retrieval-augmented generation over scientific literature
**Scope of this report:** the data pipeline (Member 1), the dense retrieval baseline (Member 2), and the PubMedQA citation-graph extension that makes graph-based retrieval evaluable.

This document is the group's **code report**, not a tutorial. It records what was built, why each design decision was taken, what the measured results are, and which limitations the reader must keep in mind. Every code block is executed against the committed artifacts, so the printed outputs are the evidence behind the claims rather than illustrations.

---

## Contents

1. Problem statement and dataset selection
2. Team responsibilities and deliverables
3. Headline results
4. Data engineering: profiling, cleaning, flattening, evidence alignment
5. Chunking: design and parameter choice
6. Executable data contract
7. Dense retrieval baseline
8. Evaluation protocol and results
9. PubMedQA citation-graph extension
10. Graph-enhanced retrieval experiment
11. Discussion and limitations
12. Reproducibility and artifact map

**Environment note.** The report runs on the committed data with `pandas`, `pyarrow`, `numpy` and `tokenizers`.
Where `sentence-transformers` and `faiss` are installed, retrieval uses the project's BGE encoder; otherwise a dependency-free hashing encoder keeps the pipeline executable. The hashing encoder measures lexical overlap only, so **every headline result in this report is quoted from the BGE-based baseline, never from the fallback**.

## 1. Problem statement and dataset selection

The project studies retrieval-augmented generation over two scientific QA benchmarks, chosen because they stress different parts of the stack.

| | QASPER v0.3 | PubMedQA-1k (expanded) |
|---|---|---|
| Domain | NLP conference papers | biomedical abstracts |
| Documents | 1,169 papers, full text | 1,395 papers (951 sources + 444 bridge references) |
| Questions | 3,598 (2,991 with gold evidence) | 1,000 (951 resolvable against OpenAlex) |
| Question style | annotator-written, often paper-bound ("they", "this paper") | self-contained scientific claims |
| Retrieval unit | paragraphs inside one paper | chunks across the whole corpus |
| Required protocol | within-paper retrieval | cross-document retrieval |

The distinction matters for the method: QASPER can only support within-document retrieval, because 36.5% of its questions contain paper-bound deixis; PubMedQA's questions are self-contained (3.3% deixis), so whole-corpus retrieval is well defined once a corpus exists. The two benchmarks therefore play complementary roles: QASPER evaluates intra-document structure, PubMedQA evaluates cross-document structure.

## 2. Team responsibilities and deliverables

| Member | Responsibility | Primary artifacts |
|---|---|---|
| Member 1 | data acquisition, cleaning, flattening, evidence alignment, chunking, data contract, validation | `data/processed/*.parquet`, `src/rag_data/`, `scripts/00–05`, `scripts/10` |
| Member 2 | dense retrieval baseline (BGE + FAISS) and retrieval evaluation | `baselines/member2_dense_rag/` |
| Members 3–5 | graph construction, GNN experiments, interactive demo | `data_process/`, `data_process/gnn_runs/`, `demo/` |

This report documents the Member 1 and Member 2 components and the citation-graph data extension; the GNN training pipeline is referenced but not re-executed here.

### Deliverables

| Artifact | Content | Rows |
|---|---|---|
| `data/processed/paragraphs.parquet` | paragraph-level coordinate system | 61,317 |
| `data/processed/qa.parquet` | evaluation gold | 3,598 |
| `data/processed/chunks/size512_ov102_soft/chunks.parquet` | default chunk table | 17,653 |
| `data/processed/pubmedqa_graph/papers.parquet` | expanded corpus | 1,395 |
| `data/processed/pubmedqa_graph/chunks.parquet` | corpus chunks | 3,707 |
| `data/processed/pubmedqa_graph/graph_edges.parquet` | citation + adjacency edges | 3,354 |

## 3. Headline results

| Result | Value |
|---|---|
| Contract validation (QASPER default config) | **19 PASS / 0 WARN / 0 FAIL** |
| QASPER dense baseline, dev (890 questions) | recall@1 **33.70%**, recall@10 **94.37%**, MRR **0.6006** |
| QASPER chunking ceiling, dev | oracle recall@1 **88.42%**, @5 **99.99%** |
| PubMedQA-graph corpus | **1,395 papers / 3,707 chunks** |
| PubMedQA-graph edges | **1,042 citations + 2,312 adjacency** |
| Cross-document coverage | **423 / 951 (44%)** of source papers have a citation bridge |
| 1-hop graph expansion (dependency-free encoder) | ~90 rankings changed, **0 newly recovered** |

The dense baseline is precision-bound: recall@10 is close to the ceiling while recall@1 is far from it, which identifies ranking rather than candidate generation as the next lever.
The graph experiment is a negative result on the portable encoder and must be repeated with BGE before any conclusion about the graph is drawn.

## 4. Data engineering

### 4.1 Profiling

`scripts/00_profile_dataset.py` reads the raw QASPER release without modifying it and records corpus size, question counts and token statistics. This establishes the denominators used later.

### 4.2 Cleaning and flattening

QASPER stores each section as a list of paragraphs with inline placeholders such as `BIBREF0` and `TABLE0`. `scripts/01_prepare_corpus.py`:

* produces `text_clean` by removing placeholders while preserving `text_raw`;
* concatenates all paragraphs of a paper into one flat string and records `char_start` / `char_end` per paragraph;
* stores an `offset_map` from cleaned to raw characters so citations can be traced back.

### 4.3 Evidence alignment

`scripts/02_align_evidence.py` maps the annotated evidence — section index, paragraph index and character span — onto `para_id`s. Questions that cannot be mapped are classified with `zero_gold_reason ∈ {no_evidence, float_only, align_fail}` and excluded from the recall denominator; they are never scored as zero.

The cell below reports the cleaning effect and the alignment funnel on the real data.

In [1]:
import json
import os
import re
import sys
import time
import zlib
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(".").resolve()
print("working directory:", ROOT)
print("python:", sys.version.split()[0], "| pandas:", pd.__version__, "| numpy:", np.__version__)

FILES = {
    "QASPER paragraphs": "data/processed/paragraphs.parquet",
    "QASPER questions": "data/processed/qa.parquet",
    "QASPER chunks": "data/processed/chunks/size512_ov102_soft/chunks.parquet",
    "QASPER gold map": "data/processed/chunks/size512_ov102_soft/qa_with_gold.jsonl",
    "PubMedQA graph chunks": "data/processed/pubmedqa_graph/chunks.parquet",
    "PubMedQA graph questions": "data/processed/pubmedqa_graph/qa.parquet",
    "PubMedQA graph edges": "data/processed/pubmedqa_graph/graph_edges.parquet",
    "PubMedQA graph papers": "data/processed/pubmedqa_graph/papers.parquet",
    "BGE tokenizer": "data/raw/bge_tok/tokenizer.json",
}
for name, p in FILES.items():
    print(f"{name:26s} {'OK' if os.path.exists(p) else 'MISSING':7s} {p}")

try:
    import faiss  # noqa: F401
    import sentence_transformers  # noqa: F401
    BACKEND = "bge"
except Exception:
    BACKEND = "hashing"
print("\nretrieval backend for this execution:", BACKEND)

working directory: D:\Users\NTU\6405\project\RAG_Project
python: 3.13.14 | pandas: 3.1.0 | numpy: 2.5.4
QASPER paragraphs          OK      data/processed/paragraphs.parquet
QASPER questions           OK      data/processed/qa.parquet
QASPER chunks              OK      data/processed/chunks/size512_ov102_soft/chunks.parquet
QASPER gold map            OK      data/processed/chunks/size512_ov102_soft/qa_with_gold.jsonl
PubMedQA graph chunks      OK      data/processed/pubmedqa_graph/chunks.parquet
PubMedQA graph questions   OK      data/processed/pubmedqa_graph/qa.parquet
PubMedQA graph edges       OK      data/processed/pubmedqa_graph/graph_edges.parquet
PubMedQA graph papers      OK      data/processed/pubmedqa_graph/papers.parquet
BGE tokenizer              OK      data/raw/bge_tok/tokenizer.json

retrieval backend for this execution: hashing


### 4.4 Corpus statistics

The table below is computed from the committed tables; it is the reference point for every denominator used later.

In [2]:
pa = pd.read_parquet("data/processed/paragraphs.parquet")
qa = pd.read_parquet("data/processed/qa.parquet")
ck = pd.read_parquet("data/processed/chunks/size512_ov102_soft/chunks.parquet")
g_ck = pd.read_parquet("data/processed/pubmedqa_graph/chunks.parquet")
g_qa = pd.read_parquet("data/processed/pubmedqa_graph/qa.parquet")
g_pa = pd.read_parquet("data/processed/pubmedqa_graph/papers.parquet")
g_ed = pd.read_parquet("data/processed/pubmedqa_graph/graph_edges.parquet")

rows = [
    ("QASPER", qa.paper_id.nunique(), len(qa), len(pa), int(pa.n_tokens.sum()), len(ck)),
    ("PubMedQA graph", g_pa.paper_id.nunique(), len(g_qa), len(g_pa), int(g_pa.abstract.str.len().sum()), len(g_ck)),
]
print(f"{'dataset':<16}{'papers':>8}{'questions':>11}{'paragraphs':>12}{'tokens/chars':>14}{'chunks':>8}")
for r in rows:
    print(f"{r[0]:<16}{r[1]:>8}{r[2]:>11}{r[3]:>12}{r[4]:>14}{r[5]:>8}")
print("\nQASPER eligible denominator (zero_gold_reason == ''):", int((qa.zero_gold_reason == "").sum()))
print("PubMedQA graph questions with gold:", int(g_qa.gold_chunk_ids.str.len().gt(0).sum()))

dataset           papers  questions  paragraphs  tokens/chars  chunks
QASPER              1169       3598       61317       6433441   17653
PubMedQA graph      1395        951        1395       1771804    3707

QASPER eligible denominator (zero_gold_reason == ''): 2991
PubMedQA graph questions with gold: 951


## 5. Chunking design

### 5.1 Coordinate systems

Three coordinate systems coexist and must not be mixed:

```
paper flat text  --char_start/char_end-->  chunk
   └-- paragraph text_clean --para_spans[a:b]--> the fragment inside a chunk
```

* `para_id = "{paper_id}#p{idx:04d}"` is globally stable and independent of the chunking configuration.
* `chunk_id = "{paper_id}#s{sec}#c{idx:03d}"` is stable **only within one configuration**.
* `para_spans` offsets are relative to the paragraph's own text, not to the chunk. In parquet they are stored as JSON text because Arrow cannot infer a `(str,int,int)` tuple column.

### 5.2 Algorithm

`src/rag_data/chunker.py` splits paragraphs into sentences, greedily packs them up to a 512-token budget, backs up whole sentences to create 20% overlap (≈102 tokens), hard-splits any single sentence that exceeds the budget (`is_truncated=True`), and records section boundaries. Section boundaries may be `hard` (never crossed) or `soft` (allowed, recorded in `n_sections`).

### 5.3 Parameter choice

The 24-point grid in `scripts/04_chunk_grid.py` selected `size=512 / overlap=0.2 / soft boundary` as the default: it keeps 94.54% of answer spans in the same chunk as their evidence, while limiting index amplification to 1.315×. Hard boundaries were rejected because they increase chunk count (23,851 vs 17,653) without improving the ceiling.

The cell below verifies the invariant on all chunks and reproduces the packing behaviour on a real paragraph.

In [3]:
violations = 0
well_formed = 0
for r in ck.itertuples():
    spans = json.loads(r.para_spans)
    if len(spans) != len(r.para_ids):
        violations += 1
    if all(len(s) == 3 for s in spans):
        well_formed += 1

print("chunks checked:", len(ck))
print("para_spans/para_ids length violations:", violations)
print("well-formed (pid, start, end) triples:", well_formed)

sample = ck.iloc[0]
spans = json.loads(sample.para_spans)
print("\nverification example:", sample.chunk_id, "| section:", sample.section_name, "| tokens:", int(sample.n_tokens))
print("  paragraphs:", list(sample.para_ids))
print("  spans:", spans[:3], "..." if len(spans) > 3 else "")
print("  flat-text character range:", (int(sample.char_start), int(sample.char_end)))

from tokenizers import Tokenizer

TOK = Tokenizer.from_file("data/raw/bge_tok/tokenizer.json")
SENT_RE = re.compile(r"(?<=[.!?])\s+")


def split_sentences(text):
    spans, start = [], 0
    for m in SENT_RE.finditer(text):
        if m.start() > start:
            spans.append((start, m.start()))
        start = m.end()
    if start < len(text):
        spans.append((start, len(text)))
    return spans or [(0, len(text))]


def pack_chunks(text, max_tokens=512, overlap=102):
    sents = split_sentences(text)
    out, i, n = [], 0, len(sents)
    while i < n:
        a = sents[i][0]
        if len(TOK.encode(text[a:sents[i][1]]).ids) > max_tokens:
            lo, hi = a, sents[i][1]
            while hi - lo > 1:
                mid = (lo + hi) // 2
                if len(TOK.encode(text[a:mid]).ids) <= max_tokens:
                    lo = mid
                else:
                    hi = mid
            b = lo
            out.append((a, b, len(TOK.encode(text[a:b]).ids), True))
            sents[i] = (b, sents[i][1])
            if b >= sents[i][1]:
                i += 1
            continue
        j = i
        while j + 1 < n and len(TOK.encode(text[a:sents[j + 1][1]]).ids) <= max_tokens:
            j += 1
        b = sents[j][1]
        out.append((a, b, len(TOK.encode(text[a:b]).ids), False))
        if j + 1 >= n:
            break
        k = j
        while k > i and len(TOK.encode(text[sents[k][0]:b]).ids) < overlap:
            k -= 1
        i = max(k, i + 1)
    return out


long_p = pa[pa.n_tokens > 512].iloc[0]
print(f"\npacking verification on {long_p.para_id} ({long_p.n_tokens} tokens)")
for a, b, t, trunc in pack_chunks(long_p.text_clean):
    print(f"  [{a:5d},{b:5d}) {t:3d} tok {'[hard-split]' if trunc else ''} | {long_p.text_clean[a:b][:52]}...")

print("\nchunk table statistics (default config)")
print("chunks:", len(ck), "| papers:", ck.paper_id.nunique(),
      f"| chunks/paper: {len(ck)/ck.paper_id.nunique():.2f}")
q = ck.n_tokens.quantile([0.5, 0.9, 0.95, 1.0]).astype(int)
print("n_tokens p50/p90/p95/max:", list(q.values))
print("abstract chunks:", int(ck.is_abstract.sum()), f"({ck.is_abstract.mean()*100:.1f}%)")
print("hard-split chunks:", int(ck.is_truncated.sum()))

chunks checked: 17653
para_spans/para_ids length violations: 0
well-formed (pid, start, end) triples: 17653

verification example: 1503.00841#sabs#c000 | section: Abstract | tokens: 496
  paragraphs: ['1503.00841#p0000', '1503.00841#p0001', '1503.00841#p0002', '1503.00841#p0003', '1503.00841#p0004']
  spans: [['1503.00841#p0000', 0, 606], ['1503.00841#p0001', 0, 406], ['1503.00841#p0002', 0, 677]] ...
  flat-text character range: (0, 2577)

packing verification on 1910.14497#p0025 (888 tokens)
  [    0, 1548) 512 tok [hard-split] | 'accountant', 'acquaintance', 'actor', 'actress', 'a...
  [ 1548, 2690) 380 tok  | , 'marshal', 'mathematician', 'mechanic', 'midfielde...

chunk table statistics (default config)
chunks: 17653 | papers: 1169 | chunks/paper: 15.10
n_tokens p50/p90/p95/max: [np.int64(495), np.int64(509), np.int64(511), np.int64(512)]
abstract chunks: 1176 (6.7%)
hard-split chunks: 29


## 6. Executable data contract

The written contract in `docs/contract/data_contract.md` is enforced by `src/rag_data/validate.py`, which implements 19 machine checks across four groups:

| Group | Checks |
|---|---|
| Structure | required columns, no extra/leaking columns, unique and well-formed ids, in-paper/in-section ordering, neighbour closure, token budget, non-empty text |
| Completeness | `para_spans` paired and unique, offsets in bounds, paragraph coverage, content coverage, `text_sha1`, `config_hash` |
| Semantics | gold paragraph references resolve, gold chunk references resolve, every question with gold maps to at least one chunk |
| Leakage | no answer-derived fields in public tables |

Any `FAIL` blocks release of the artifact. The cell runs the same validator used by `scripts/05_validate_contract.py`.

In [4]:
sys.path.insert(0, "src")
from rag_data import validate as V

rows_v = []
for r in ck.to_dict("records"):
    s = r["para_spans"]
    r["para_spans"] = [tuple(x) for x in json.loads(s)] if isinstance(s, str) else s
    rows_v.append(r)

para_len = dict(zip(pa.para_id, pa.n_chars))
para_text = dict(zip(pa.para_id, pa.text_clean))
qrows = [json.loads(line) for line in open(FILES["QASPER gold map"], encoding="utf-8")]

findings = []
findings += V.check_columns(rows_v, V.CHUNK_REQUIRED, V.FORBIDDEN_IN_PUBLIC)
findings += V.check_chunk_ids(rows_v)
findings += V.check_ordering(rows_v)
findings += V.check_neighbours(rows_v)
findings += V.check_token_budget(rows_v, 512)
findings += V.check_para_spans(rows_v, para_len)
findings += V.check_coverage(rows_v, para_len, para_text)
findings += V.check_no_gold_in_public(rows_v)
findings += V.check_gold(qrows, set(ck.chunk_id), set(pa.para_id))

for f in findings:
    print(f"  {f.status:4s} {f.check}")
print("\nsummary (pass, warn, fail):", V.summarise(findings))

  PASS columns.required
  PASS columns.extra
  PASS columns.leakage
  PASS id.unique
  PASS id.format
  PASS order.in_paper
  PASS order.in_section
  PASS graph.neighbours
  PASS size.within_budget
  PASS size.non_empty
  PASS spans.paired
  PASS spans.unique
  PASS spans.in_bounds
  PASS coverage.paragraphs
  PASS coverage.content
  PASS leakage.value_fields
  PASS gold.para_refs
  PASS gold.chunk_refs
  PASS gold.reachable

summary (pass, warn, fail): (19, 0, 0)


## 7. Dense retrieval baseline

Member 2 implements the baseline in `baselines/member2_dense_rag/`:

* encoder `BAAI/bge-small-en-v1.5` (fixed revision), documents encoded directly, queries prefixed with the retrieval instruction;
* L2-normalised vectors indexed with FAISS `IndexFlatIP`, i.e. exact cosine search;
* `scope="paper"` for QASPER (candidates restricted to the question's paper) and global search for PubMedQA-graph;
* a `preview` generator that never calls an LLM, so the reported numbers are pure retrieval.

The report reproduces the retrieval mechanics with a portable encoder so that the loop and metrics remain verifiable without GPU dependencies. The **headline numbers below are read from Member 2's BGE run**, not from this fallback.

In [5]:
DIM = 4096
_encoder = None


def _tokens(s):
    return re.findall(r"[a-z0-9]+", s.lower())


def encode(texts):
    global _encoder
    if BACKEND == "bge" and _encoder is None:
        from sentence_transformers import SentenceTransformer
        _encoder = SentenceTransformer("BAAI/bge-small-en-v1.5")
    if BACKEND == "bge":
        return _encoder.encode(texts, normalize_embeddings=True, batch_size=32)
    m = np.zeros((len(texts), DIM), dtype=np.float32)
    for i, t in enumerate(texts):
        for w in _tokens(t):
            m[i, zlib.crc32(w.encode()) % DIM] += 1.0
        norm = np.linalg.norm(m[i])
        if norm:
            m[i] /= norm
    return m


t0 = time.time()
C = encode(g_ck.text.tolist())
print("chunk matrix:", C.shape, f"| encoded in {time.time()-t0:.1f}s | backend={BACKEND}")

qid = "21645374"
qtext = g_qa[g_qa.question_id == qid].iloc[0].question
qv = encode([qtext])[0]
scores = C @ qv
top = np.argsort(-scores)[:5]

print("\nquery:", qtext)
print("gold :", list(g_qa[g_qa.question_id == qid].iloc[0].gold_chunk_ids))
for rank, idx in enumerate(top, 1):
    r = g_ck.iloc[idx]
    print(f"  #{rank} {r.chunk_id} | {r.section_name:<12s} | score {scores[idx]:.3f}")
    print(f"      {r.text[:76]}...")

chunk matrix: (3707, 4096) | encoded in 0.2s | backend=hashing

query: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
gold : ['W2018063873#s00#c000', 'W2018063873#s01#c000']
  #1 W2082571460#s00#c000 | PURPOSE      | score 0.324
      A possible role for fondaparinux as a bridging agent in the perioperative se...
  #2 W2018063873#s00#c000 | BACKGROUND   | score 0.322
      Programmed cell death (PCD) is the regulated death of cells within an organi...
  #3 W2018063873#s01#c000 | RESULTS      | score 0.274
      The following paper elucidates the role of mitochondrial dynamics during dev...
  #4 W2094548557#s04#c000 | MAIN OUTCOME MEASURE | score 0.267
      Death....
  #5 W2262039363#s00#c000 | BACKGROUND   | score 0.260
      Anchoring vignettes are brief texts describing a hypothetical character who ...


## 8. Evaluation protocol and results

### 8.1 Protocol

* **QASPER** is scored at **paragraph level**: retrieved chunks are expanded to `para_ids`, intersected with `gold_para_ids`, divided by the number of gold paragraphs, then macro-averaged over questions. The denominator is the 2,991 questions with gold overall, of which **890 are in dev**; questions without gold are excluded, not zero-scored. Chunk-level recall is never used across configurations because chunk ids are configuration-specific.
* **PubMedQA-graph** is scored at the **paper level**: the pool is the whole corpus and gold is the set of chunks belonging to the source paper.

$$\mathrm{Hit@}k=\frac{1}{N}\sum_i \mathbb{1}\left[\text{top-}k_i \cap \text{gold}_i \neq \varnothing\right],
\qquad
\mathrm{MRR}=\frac{1}{N}\sum_i \frac{1}{\text{rank}_i}$$

In [6]:
gold = {r.question_id: set(r.gold_chunk_ids) for r in g_qa.itertuples()}
chunk_ids = g_ck.chunk_id.tolist()
id2idx = {c: i for i, c in enumerate(chunk_ids)}
qids = list(gold)

Q = encode(g_qa.question.fillna("").tolist())
S = Q @ C.T
print("score matrix:", S.shape)


def hit_at(k, ranking):
    return float(np.mean([bool(set(ranking[i][:k]) & gold[q]) for i, q in enumerate(qids)]))


def mrr_at(k, ranking):
    vals = []
    for i, q in enumerate(qids):
        rr = 0.0
        for pos, c in enumerate(ranking[i][:k], 1):
            if c in gold[q]:
                rr = 1.0 / pos
                break
        vals.append(rr)
    return float(np.mean(vals))


order = np.argsort(-S, axis=1)[:, :10]
rk = [[chunk_ids[j] for j in row] for row in order]

print(f"questions: {len(qids)} | backend: {BACKEND}")
print(f"Hit@1 {hit_at(1, rk):.4f} | Hit@3 {hit_at(3, rk):.4f} | Hit@10 {hit_at(10, rk):.4f} | MRR {mrr_at(10, rk):.4f}")
print("\nThese values come from the portable encoder and are diagnostic only.")

score matrix: (951, 3707)
questions: 951 | backend: hashing
Hit@1 0.6856 | Hit@3 0.7771 | Hit@10 0.8475 | MRR 0.7373

These values come from the portable encoder and are diagnostic only.


### 8.2 QASPER results

Member 2's BGE run on dev:

| k | Recall@k | MRR@k |
|---|---|---|
| 1 | 33.70% | 0.4213 |
| 3 | 63.00% | 0.5569 |
| 5 | 77.33% | 0.5836 |
| 10 | 94.37% | 0.6006 |

The **chunking ceiling** (`oracle_recall@k`, recomputed below) is the model-free upper bound imposed by how evidence is distributed across chunks. A method exceeding it is necessarily miscalculated.

In [7]:
dev = qa[(qa.split == "dev") & (qa.zero_gold_reason == "")].copy()
paper_chunks = {}
for r in ck.itertuples():
    paper_chunks.setdefault(r.paper_id, []).append(set(r.para_ids))

KS = (1, 3, 5, 10)
acc = {k: 0.0 for k in KS}
n = 0
for r in dev.itertuples():
    G = set(r.gold_para_ids)
    if not G:
        continue
    cands = paper_chunks.get(r.paper_id, [])
    chosen, covered, step = set(), set(), {}
    for s in range(1, max(KS) + 1):
        best, best_new = None, -1
        for i, c in enumerate(cands):
            if i in chosen:
                continue
            new = len((covered | c) & G)
            if new > best_new:
                best_new, best = new, i
        if best is None:
            break
        chosen.add(best)
        covered |= cands[best]
        step[s] = len(covered & G)
        if len(covered & G) == len(G):
            break
    run = 0
    for k in KS:
        for s in range(1, k + 1):
            run = max(run, step.get(s, 0))
        acc[k] += run / len(G)
    n += 1

print(f"dev questions with gold: {n}")
print("                 k=1     k=3     k=5    k=10")
print("oracle recall  " + "  ".join(f"{acc[k]/n*100:6.2f}" for k in KS))

p = Path("baselines/member2_dense_rag/docs/qasper_dev_metrics.json")
if p.exists():
    d = json.loads(p.read_text(encoding="utf-8"))
    print(f"\nMember 2 dense baseline | split={d['split']} | total={d['total_questions']} | eligible={d['eligible_questions']}")
    print("                Recall@k     MRR")
    for row in d["metrics"]:
        print(f"  k={row['k']:<2d}          {row['macro_paragraph_recall']*100:6.2f}%   {row['mrr']:.4f}")
else:
    print("\nMember 2 metrics file not found")

dev questions with gold: 890
                 k=1     k=3     k=5    k=10
oracle recall   88.42   99.76   99.99  100.00

Member 2 dense baseline | split=dev | total=1005 | eligible=890
                Recall@k     MRR
  k=1            33.70%   0.4213
  k=3            63.00%   0.5569
  k=5            77.33%   0.5836
  k=10           94.37%   0.6006


## 9. PubMedQA citation-graph extension

### 9.1 Motivation

CG-RAG requires three ingredients: a graph with edges, a retrieval pool, and questions that do not depend on a specific paper. The original PubMedQA provides none of them: every paper has a single abstract chunk (no pool), papers carry no citation information (no graph), and the task reduces to reading the provided abstract.

### 9.2 Construction

| Stage | Operation | Outcome |
|---|---|---|
| P0 | resolve 1,000 PMIDs on OpenAlex | 951 resolved (49 not indexed) |
| P1 | collect `referenced_works` | 24,457 directed citation edges |
| P2 | expand corpus with references | 23,792 unique references |
| P2' | keep references cited ≥ 2 times | 444 bridge papers |
| P3 | chunk the corpus (`CONTEXTS`, abstracts) | 1,395 papers → 3,707 chunks |
| P4 | build edges | 1,042 citations + 2,312 adjacency |
| P5 | redefine QA | gold = source paper's chunks, 951 questions |

### 9.3 Design decision: the coupling core

The one-hop citation graph is a **near-star**: of 23,792 references, only 448 are cited more than once, and only 7 citation edges fall inside the source set. Keeping all references would triple the corpus, add 41% title-only nodes, and contribute no cross-document structure. Retaining only the references cited at least twice keeps every added paper as a genuine bridge between at least two source papers, at the cost of a smaller pool (1,395 instead of ~24,700).

The trade-off is reported explicitly: **423 of 951 source papers (44%)** have at least one citation edge, so graph effects can only appear on that subset.

In [8]:
print("corpus composition")
print(g_pa.role.value_counts().to_string())
print("\nchunks per role:")
print(g_ck.merge(g_pa[["paper_id", "role"]], on="paper_id").role.value_counts().to_string())
print("\nabstract coverage by role (%):")
print((g_pa.groupby("role").has_abstract.mean() * 100).round(1).to_string())

print("\ngraph edges")
print(g_ed.edge_type.value_counts().to_string())
cites = g_ed[g_ed.edge_type == "cites"]
deg_out = cites.groupby("src_id").size()
deg_in = cites.groupby("dst_id").size()
print(f"\ncitations: {len(cites)} edges | papers with outgoing: {deg_out.size} | with incoming: {deg_in.size}")
print("most cited bridge papers:")
for pid, d in deg_in.sort_values(ascending=False).head(5).items():
    title = g_pa.loc[g_pa.paper_id == pid, "title"].iloc[0]
    print(f"  {pid}  cited by {d:2d}  | {title[:58]}")

pid = "W2147896631"
print("\nbridged example:", pid, "|", g_pa.loc[g_pa.paper_id == pid, "title"].iloc[0])
print("question:", g_qa[g_qa.source_paper_id == pid].question.iloc[0])
print("citation neighbours in corpus:")
for x in g_pa[g_pa.paper_id.isin(cites[cites.src_id == pid].dst_id)].head(6).itertuples():
    print(f"  -> {x.paper_id} | {x.title[:62]}")

pid2 = "W2018063873"
print("\nunbridged counter-example:", pid2, "|", g_pa.loc[g_pa.paper_id == pid2, "title"].iloc[0])
print("  outgoing citations in corpus:", int((cites.src_id == pid2).sum()),
      "| incoming:", int((cites.dst_id == pid2).sum()))

corpus composition
role
source       951
reference    444

chunks per role:
role
source       3192
reference     515

abstract coverage by role (%):
role
reference     67.8
source       100.0

graph edges
edge_type
adjacency    2312
cites        1042

citations: 1042 edges | papers with outgoing: 425 | with incoming: 451
most cited bridge papers:
  W3147894994  cited by 10  | Regression Models and Life-Tables
  W1979300931  cited by 10  | Nonparametric Estimation from Incomplete Observations
  W2159011576  cited by  9  | Diagnostic and Statistical Manual of Mental Disorders
  W2166281097  cited by  9  | The Hospital Anxiety and Depression Scale
  W2164777277  cited by  8  | The Measurement of Observer Agreement for Categorical Data

bridged example: W2147896631 | Is Nasogastric Decompression Useful in Prevention of Leaks After Laparoscopic Sleeve Gastrectomy? A Randomized Trial
question: Is nasogastric decompression useful in prevention of leaks after laparoscopic sleeve gastrectomy?
c

## 10. Graph-enhanced retrieval experiment

Before training a graph encoder, we test the mechanism with a training-free variant:

1. dense top-10 → **seed chunks**;
2. seed chunk → paper → **one-hop citation neighbours**;
3. neighbour chunks enter the candidate pool;
4. rescore: $\text{score}(q,c)=\text{sim}(q,c)+\lambda\cdot\max_{s\in\text{seeds}}\text{sim}(c,s)$.

This ordering is deliberate: a cheap negative result justifies not investing in LeSeGR-style message passing; a positive result justifies training it.

### Result

On the portable encoder the expansion changes roughly 90 of 951 rankings but recovers **zero** additional gold chunks. The mechanism executed correctly (candidate pool 50 → 92); it simply had nothing to add on a lexical encoder, where the gold abstract is already retrieved by term overlap. The experiment must be repeated with BGE before drawing conclusions about the graph.

In [9]:
paper_of = dict(zip(g_ck.chunk_id, g_ck.paper_id))
chunks_of = {}
for r in g_ck.itertuples():
    chunks_of.setdefault(r.paper_id, []).append(r.chunk_id)

nbr = {}
for r in cites.itertuples():
    nbr.setdefault(r.src_id, set()).add(r.dst_id)
    nbr.setdefault(r.dst_id, set()).add(r.src_id)

LAM = 0.5
BASE_K = 50


def expand_rank(i, topk=10, base_k=BASE_K):
    seed = [chunk_ids[j] for j in np.argsort(-S[i])[:topk]]
    pool = [chunk_ids[j] for j in np.argsort(-S[i])[:base_k]]
    papers = {paper_of[c] for c in seed}
    cand_papers = set(papers)
    for p in papers:
        cand_papers |= nbr.get(p, set())
    cand = {c for p in cand_papers for c in chunks_of.get(p, [])} | set(pool)
    ci = np.array([id2idx[c] for c in cand])
    base = S[i, ci]
    si = np.array([id2idx[c] for c in seed])
    neigh = (C[ci] @ C[si].T).max(axis=1)
    total = base + LAM * neigh
    order = ci[np.argsort(-total)]
    return [chunk_ids[j] for j in order[:topk]], len(cand)


t0 = time.time()
rk2, sizes = [], []
for i in range(len(qids)):
    r, n_cand = expand_rank(i)
    rk2.append(r)
    sizes.append(n_cand)

changed = sum(1 for a, b in zip(rk, rk2) if a != b)
recovered = sum(
    1 for i, q in enumerate(qids)
    if not (set(rk[i][:10]) & gold[q]) and (set(rk2[i][:10]) & gold[q])
)
print(f"expansion: {time.time()-t0:.1f}s | mean candidates {np.mean(sizes):.0f} (dense pool {BASE_K})")
print(f"rankings changed: {changed} | newly recovered gold: {recovered}")
print()
print("          Hit@1    Hit@3    Hit@10     MRR")
print(f"dense   {hit_at(1,rk):7.4f} {hit_at(3,rk):8.4f} {hit_at(10,rk):9.4f} {mrr_at(10,rk):7.4f}")
print(f"1-hop   {hit_at(1,rk2):7.4f} {hit_at(3,rk2):8.4f} {hit_at(10,rk2):9.4f} {mrr_at(10,rk2):7.4f}")

expansion: 1.6s | mean candidates 92 (dense pool 50)
rankings changed: 94 | newly recovered gold: 0

          Hit@1    Hit@3    Hit@10     MRR
dense    0.6856   0.7771    0.8475  0.7373
1-hop    0.6856   0.7771    0.8475  0.7375


## 11. Discussion and limitations

### 11.1 What the results establish

1. **The data pipeline is sound.** All 19 contract checks pass, including span integrity, neighbour closure, token budget, paragraph coverage and gold reachability.
2. **The dense baseline is precision-bound.** Recall@10 reaches 94.37% against a ceiling of 100%, while recall@1 reaches only 33.70% against a ceiling of 88.42%. Evidence is retrieved but not ranked first, which points to reranking and hybrid retrieval as the next interventions.
3. **PubMedQA can support cross-document retrieval, but only after expansion.** The corpus now has a pool (3,707 chunks), a graph (1,042 citations), and self-contained questions; without the expansion none of the three existed.

### 11.2 Limitations

* **QASPER is single-document by construction.** 36.5% of its questions contain paper-bound deixis; a whole-corpus setting is undefined for them, so cross-document graph methods cannot be evaluated on QASPER.
* **The PubMedQA graph is sparse.** Only 44% of source papers have a citation edge; results must be reported separately for the bridged and unbridged subsets.
* **PubMedQA-graph is abstract-level.** One paper is ~2.7 chunks, so this measures paper retrieval more than paragraph retrieval; full text would change the granularity.
* **No generation evaluation.** The generator is a `preview` stub; answer quality (yes/no/maybe) is out of scope for this report.
* **External numbers are not comparable.** The reference implementation's 7,849-paper graph and its Hit@1 = 0.961 rely on an unpublished reference-extraction policy. All comparisons in this report are within our own pool.

### 11.3 Denominators that must not be mixed

| Benchmark | Eligible denominator | Why |
|---|---|---|
| QASPER, all | 2,991 | 607 questions have no usable paragraph evidence |
| QASPER, dev | 890 | dev subset of the eligible questions |
| PubMedQA-graph | 951 | 49 PMIDs are absent from OpenAlex |

## 12. Reproducibility and artifact map

```bash
# rebuild the data from raw QASPER
python scripts/00_profile_dataset.py --data-dir data/raw/qasper --out-dir docs
python scripts/01_prepare_corpus.py  --data-dir data/raw/qasper --processed-dir data/processed
python scripts/02_align_evidence.py  --data-dir data/raw/qasper --processed-dir data/processed
python scripts/03_build_chunks.py    --chunk-size 512 --overlap 0.2 --section-boundary soft
python scripts/05_validate_contract.py

# rebuild the PubMedQA citation graph
python scripts/20_expand_pubmedqa_corpus.py --min-cited 2
python scripts/21_build_pubmedqa_graph.py

# tests
python -m pytest tests -q
```

| Component | Location |
|---|---|
| Data contract (authoritative) | `docs/contract/data_contract.md` |
| Chunker | `src/rag_data/chunker.py` |
| Contract validator | `src/rag_data/validate.py` |
| Dense baseline (BGE + FAISS) | `baselines/member2_dense_rag/` |
| Citation-graph pipeline | `scripts/20_*.py`, `scripts/21_*.py` |
| GNN experiments | `data_process/`, `data_process/gnn_runs/` |
| Interactive demo | `demo/` |